# M6S2 · Exascale computing: why data movement matters

**IE University | HPC 2026–27 | About 30 minutes | C code on one CPU**

## The question
An exascale supercomputer performs an enormous number of calculations. Why can a program still be slow or consume significant energy? Because moving data through memory and between processors has a cost.

**Your mission:** Run two correct C implementations of the same simple calculation, compare their *logical data movement*, and explain the trade-off. You do not need GPUs, MPI, quantum hardware or a large dataset.

This is a **teaching model**, not an energy benchmark. The counted bytes are the loads and stores in the algorithm, **not measured DRAM traffic**: caches, compilers and hardware can change actual physical movement.

By the end, you will understand (1) the meaning of exascale, (2) memory movement versus arithmetic, (3) why software design matters, and (4) when an optimization is mathematically valid.


## 1. Predict first

Imagine 100,000 sensor values. We add 1 to every value, eight times.

- **Approach A:** make eight full passes over the array, reading and writing all values on each pass.
- **Approach B:** since the operation is independent for every value, make one pass and add 8 once.

**Before running:** Are the answers the same? Which one carries out more array reads and writes? What would change if every update depended on a neighbouring value?

## 2. Run the program

The following Python cell only **writes, compiles and launches a C program**. The parallel-computing example itself is entirely C. Use JupyterLab with a Python 3 kernel and GCC. If you work on SciTech and Slurm is available, the notebook uses one small srun CPU allocation. Do not run large compute jobs on a login node.


In [ ]:
C_SOURCE = r'''
#include <math.h>
#include <stdio.h>
#include <stdlib.h>
#include <string.h>

int main(int argc, char **argv) {
    const size_t n = 100000;          /* small teaching array */
    int steps = (argc > 1) ? atoi(argv[1]) : 8;
    if (steps < 1 || steps > 100) {
        fprintf(stderr, "Choose 1 to 100 steps.\n");
        return 2;
    }

    double *initial = malloc(n * sizeof(double));
    double *a = malloc(n * sizeof(double));
    double *b = malloc(n * sizeof(double));
    double *fused = malloc(n * sizeof(double));
    if (!initial || !a || !b || !fused) {
        fprintf(stderr, "Allocation failed.\n");
        free(initial); free(a); free(b); free(fused);
        return 3;
    }

    for (size_t i = 0; i < n; i++) {
        initial[i] = (double)(i % 13);
        a[i] = initial[i];
    }

    /* Approach A: each step reads and writes the full array. */
    for (int step = 0; step < steps; step++) {
        for (size_t i = 0; i < n; i++) b[i] = a[i] + 1.0;
        double *tmp = a; a = b; b = tmp;
    }

    /* Approach B: algebraically equivalent, one pass. */
    for (size_t i = 0; i < n; i++) {
        fused[i] = initial[i] + (double)steps;
    }

    double max_error = 0.0;
    for (size_t i = 0; i < n; i++) {
        double error = fabs(a[i] - fused[i]);
        if (error > max_error) max_error = error;
    }

    /* Logical algorithmic data movement: one read + one write per value/pass.
       These are not observed memory-system transfers or energy measurements. */
    double pass_bytes = 2.0 * (double)n * (double)sizeof(double);
    double repeated_mb = pass_bytes * (double)steps / 1.0e6;
    double fused_mb = pass_bytes / 1.0e6;

    printf("N=%zu values, STEPS=%d\n", n, steps);
    printf("Repeated passes: %.2f MB logical reads+writes\n", repeated_mb);
    printf("Fused pass:      %.2f MB logical reads+writes\n", fused_mb);
    printf("Logical traffic ratio: %.1f x\n", repeated_mb / fused_mb);
    printf("max_abs_difference=%.0f\n", max_error);
    printf("Result at index 42: repeated=%.0f fused=%.0f\n", a[42], fused[42]);

    free(initial); free(a); free(b); free(fused);
    return max_error == 0.0 ? 0 : 4;
}
'''


In [ ]:
from pathlib import Path
import os, shutil, subprocess, tempfile

work = Path(tempfile.mkdtemp(prefix="m6s2_exascale_"))
source = work / "data_movement.c"
source.write_text(C_SOURCE)
if shutil.which("gcc") is None:
    print("SKIPPED: GCC not available. Load your course compiler and rerun.")
else:
    compile_job = subprocess.run(
        ["gcc", "-std=c11", "-O2", "-Wall", "-Wextra", "-Werror", str(source), "-lm", "-o", str(work / "demo")],
        capture_output=True, text=True,
    )
    if compile_job.returncode != 0:
        raise RuntimeError("FAIL: compilation\n" + compile_job.stderr)
    print("PASS: C compilation")

def run_demo(steps=8):
    if shutil.which("gcc") is None: return None
    if not 1 <= steps <= 100: raise ValueError("Choose 1..100 steps")
    cmd = [str(work / "demo"), str(steps)]
    env = os.environ.copy()
    if shutil.which("srun"):
        for key in ("SLURM_MEM_PER_CPU", "SLURM_MEM_PER_NODE", "SLURM_MEM_PER_GPU"):
            env.pop(key, None)
        cmd = ["srun", "-p", "cpu", "-N1", "-n1", "-c1", "--mem=1G", "--time=00:02:00"] + cmd
        print("Using one small Slurm CPU allocation; a queue delay is not a result.")
    result = subprocess.run(cmd, capture_output=True, text=True, env=env)
    if result.returncode:
        raise RuntimeError("FAIL: execution\n" + result.stderr)
    print(result.stdout)
    assert "max_abs_difference=0" in result.stdout
    print("PASS: both implementations agree")
    return result.stdout

run_demo(8)


## 3. Change one number

Run the next cell. It uses **four** passes instead of eight. Compare the two traffic ratios.

**Predict:** Does the correctness check still pass? How does the logical ratio change?


In [ ]:
run_demo(4)


## 4. Interpret the result

1. Why do the two programs return the same values? Because each update is **independent** of the other array elements and adding 1 eight times equals adding 8 once.
2. Why is the fused method interesting to HPC? It avoids repeated full-array passes, so the algorithm requests fewer reads and writes.
3. Does 8× less *logical* movement mean 8× less energy? **No.** We did not measure memory-controller activity, execution time or energy; cache reuse and compiler optimizations matter.
4. What about a weather-grid stencil? Each output uses neighbouring cells. Simply replacing many physical timesteps by one bigger addition would produce **different answers**. Mathematical dependencies restrict optimizations.

**HPC connection:** Exascale systems combine CPUs, GPUs, fast memory and large networks. Their software must move data wisely, tolerate failures and scale efficiently. Peak FLOPS alone do not guarantee fast real applications.


## 5. Five quick checks (reveal answers after predicting)

**Q1.** What does one exaflop/s mean?  
A. 10¹² floating-point operations/s  B. 10¹⁸ floating-point operations/s  C. 10¹⁸ bytes/s
<details><summary>Answer</summary>B. An exaflop/s is one quintillion floating-point operations per second.</details>

**Q2.** Why can memory bandwidth limit a GPU program?  
A. Each operation needs data, and supplying it can take longer than computing.  B. GPUs cannot read arrays.  C. Every loop is automatically parallel.
<details><summary>Answer</summary>A. This is a form of the memory-wall problem.</details>

**Q3.** Our repeated and fused C programs match because...  
A. We skipped all calculations.  B. Every update is independent and the additions can be combined.  C. Physical memory has no latency.
<details><summary>Answer</summary>B. The transformation is valid only because of the operation's mathematical independence.</details>

**Q4.** Which statement about our logical-traffic number is correct?  
A. It directly measures joules.  B. It proves an 8× hardware speedup.  C. It counts algorithmic loads and stores only.
<details><summary>Answer</summary>C. Hardware and energy were not measured.</details>

**Q5.** Why do large HPC systems need fault tolerance?  
A. Many components can fail during long jobs.  B. Checkpointing always increases useful FLOPS.  C. Hardware never fails at scale.
<details><summary>Answer</summary>A. Saving progress and recovery protect long calculations.</details>


## 6. Exit ticket

Write **three short sentences**:

1. With 8 steps, the two programs moved approximately ____ MB and ____ MB of *logical* data.
2. Both programs gave the same answer because ____.
3. For a real exascale weather simulation, this simple fusion might be invalid because ____.

**Optional:** Mention one other post-exascale challenge (energy, reliability, heterogeneous CPUs/GPUs, co-design or software scalability).

**This notebook is practice, not a new graded assignment.**

### Sources
- [IE University HPC 2026–27 syllabus, Sessions 27–28] (course handout)
- [EuroHPC JU: Europe's JUPITER system](https://www.eurohpc-ju.europa.eu/supercomputers/our-supercomputers_en)
- [Oak Ridge National Laboratory: Frontier](https://www.olcf.ornl.gov/frontier/)
